In [0]:
dbutils.widgets.text("catalogo", "electrocasa_dev", "Catálogo a validar")
catalogo = dbutils.widgets.get("catalogo")
spark.sql(f"USE CATALOG {catalogo}")

tablas_obligatorias = [
    "bronze.ventas", "bronze.catalogo", "bronze.empleados", "bronze.resenas",
    "bronze.devoluciones", "bronze.tracking_envios", "bronze.sucursales",
    "silver.ventas", "silver.catalogo", "silver.empleados", "silver.resenas",
    "silver.devoluciones", "silver.tracking_envios", "silver.sucursales",
    "silver.cuarentena",
    "gold.ventas_netas_sucursal_mes", "gold.top_productos", "gold.dotacion_mensual",
    "gold.ventas_vendedor", "gold.resenas_negativas", "gold.despachos_courier",
    "gold.empleados_vigentes", "gold.resenas_negativas_origen"
]

print(f"=== VALIDACIÓN DE CALIDAD EN {catalogo} ===")
vacias = []
for t in tablas_obligatorias:
    conteo = spark.table(t).count()
    print(f"{'OK ' if conteo > 0 else 'ERR'} | {t}: {conteo} filas")
    if conteo == 0:
        vacias.append(t)

cerrados = spark.table("silver.empleados").filter("__END_AT IS NOT NULL").count()
print(f"OK  | Versiones históricas cerradas SCD2 en silver.empleados: {cerrados}")

dup_tracking = spark.sql("""
    SELECT COUNT(*) AS c FROM (
      SELECT tracking_id, estado_entrega FROM silver.tracking_envios
      GROUP BY tracking_id, estado_entrega HAVING COUNT(*) > 1)
""").first()["c"]
print(f"OK  | Estados duplicados por envío en silver.tracking_envios: {dup_tracking}")

display(spark.sql("""
    SELECT fuente, regla_fallida, COUNT(*) AS filas_rechazadas
    FROM silver.cuarentena
    GROUP BY fuente, regla_fallida
    ORDER BY filas_rechazadas DESC
"""))

assert not vacias, f"ERROR CRÍTICO: tablas vacías: {vacias}"
assert cerrados > 0, "ERROR: No se generaron intervalos históricos (__END_AT) en silver.empleados."
assert dup_tracking == 0, "ERROR: quedan mensajes reprocesados en silver.tracking_envios."

In [0]:
%sql
SELECT * FROM electrocasa_dev.gold.despachos_courier;
SELECT _source_file, COUNT(*) FROM electrocasa_dev.bronze.ventas   GROUP BY 1 ORDER BY 1;
SELECT _source_file, COUNT(*) FROM electrocasa_dev.bronze.catalogo GROUP BY 1 ORDER BY 1;

In [0]:
%sql
SELECT _source_file, COUNT(*) FROM electrocasa_dev.bronze.ventas GROUP BY 1 ORDER BY 1;

In [0]:
%sql
SELECT * FROM electrocasa_dev.gold.despachos_courier;